# OFC 2026 — PI-EDFA & Transformer Transfer Learning

**COSMOS(과거 대규모 측정) 사전학습 → 최근 Train 미세조정 → Scratch 비교**

기존 두 아키텍처의 코드/모델을 그대로 사용합니다. 초기에는 공개 TestGround를 보지 않고 최근 Train에서 분리한 검증 세트로 모델을 선택합니다. Colab GPU 런타임(T4 권장)에서 실행하세요.

공식 데이터: https://github.com/ofc-ml-challenge/ofc-ml-challenge-data-code
우리 코드: https://github.com/kimheeseo/LSCNS/tree/main/26OFC_ML_challenge/COSMOS_Transfer_Learning

In [ ]:
import sys, subprocess, torch
print('Python:',sys.version)
print('Torch:',torch.__version__, 'CUDA:',torch.cuda.is_available())
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'numpy', 'pandas', 'scikit-learn', 'matplotlib'],check=True)
subprocess.run(['bash','-lc','git lfs version >/dev/null || (apt-get update -qq && apt-get install -y -qq git-lfs)'],check=True)

## 1. 모델 코드와 데이터 준비 (공식 Git LFS CSV)

대량의 COSMOS 원본 JSON은 받지 않고 공식 저장소의 이미 변환된 COSMOS/Train CSV만 다운로드합니다. 두 저장소 모두 sparse clone을 사용합니다.

In [ ]:
from pathlib import Path
import subprocess

def sh(cmd):
    print('+', ' '.join(map(str,cmd)),flush=True)
    subprocess.run([str(v) for v in cmd],check=True)

root=Path('/content/ofc-ml-challenge-data-code')
if not (root/'.git').exists():
    sh(['bash','-lc',f'GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 --filter=blob:none --sparse https://github.com/ofc-ml-challenge/ofc-ml-challenge-data-code.git {root}'])
sh(['git','-C',str(root),'sparse-checkout','set','Features'])
sh(['git','-C',str(root),'lfs','install','--local'])
required=[
 'Features/Train/COSMOS_features.csv','Features/Train/COSMOS_labels.csv',
 'Features/Train/train_features.csv','Features/Train/train_labels.csv',
 'Features/Test/test_features.csv',
]
sh(['git','-C',str(root),'lfs','pull','--include='+','.join(required),'--exclude='])

repo=Path('/content/LSCNS')
if not (repo/'.git').exists():
    sh(['git','clone','--depth','1','--filter=blob:none','--sparse',
        'https://github.com/kimheeseo/LSCNS.git',str(repo)])
sh(['git','-C',str(repo),'sparse-checkout','set',
    '26OFC_ML_challenge/COSMOS_Transfer_Learning',
    '26OFC_ML_challenge/PI_EDFA_Transformer'])
runner=repo/'26OFC_ML_challenge/COSMOS_Transfer_Learning/train_transfer.py'
assert runner.is_file(),runner
for name in required:
    file=root/name
    assert file.exists() and file.stat().st_size>1024, f'CSV 미다운로드(LFS 포인터?): {file}'
print('코드 준비 완료:',runner)
print('COSMOS CSV 크기(MB):',round((root/required[0]).stat().st_size/1e6,1),
      round((root/required[1]).stat().st_size/1e6,1))

## 2. 학습 설정

`COSMOS_MAX_ROWS=0`이면 COSMOS 전체 샘플을 사용합니다. 속도와 GPU 메모리가 부담되면 샘플 수를 줄여 먼저 확인하세요. `RUN_TESTGROUND=False`를 유지해 모델 선택 과정에서 사후 공개 정답이 개입되지 않게 합니다.

In [ ]:
#@title 실행 파라미터
COSMOS_MAX_ROWS = 0       # 0 = 전체, 5000 = 소규모 예비 실행
PRETRAIN_EPOCHS = 60
FINETUNE_EPOCHS = 180
PRETRAIN_BATCH = 128
FINETUNE_BATCH = 64
RUN_SCRATCH_CONTROL = True
RUN_TESTGROUND = False     # 모든 모델 고정 후 사후 검증할 때만 True
USE_CPU = False
OUTPUT = '/content/ofc_cosmos_transfer_results'

if RUN_TESTGROUND:
    extra=['Features/TestGround/test_features.csv','Features/TestGround/test_labels.csv']
    sh(['git','-C',str(root),'lfs','pull','--include='+','.join(extra),'--exclude='])

## 3. COSMOS pretrain → 최근 Train fine-tune

PI-EDFA와 Transformer를 각각 사전학습·미세조정합니다. 최종 비교를 위해 최근 Train만 사용하는 scratch 모델도 같은 분할로 학습시킵니다.

In [ ]:
import sys
cmd=[sys.executable,str(runner),
     '--data-root',str(root),'--output',OUTPUT,
     '--models','pi-edfa','transformer',
     '--pretrain-epochs',str(PRETRAIN_EPOCHS),
     '--finetune-epochs',str(FINETUNE_EPOCHS),
     '--pretrain-batch-size',str(PRETRAIN_BATCH),
     '--batch-size',str(FINETUNE_BATCH),
     '--cosmos-max-rows',str(COSMOS_MAX_ROWS)]
if not RUN_SCRATCH_CONTROL:cmd+=['--skip-scratch']
if RUN_TESTGROUND:cmd+=['--evaluate-testground']
if USE_CPU:cmd+=['--cpu']
sh(cmd)

## 4. 모델별 성능 비교

다음 표는 위 학습 셀이 끝난 후 실제 계산값으로 채워집니다. 아직 실행하지 않았다면 점수는 생성되지 않습니다. 공식 점수는 낮을수록 좋습니다.

In [ ]:
import json, pandas as pd
from pathlib import Path
from IPython.display import display
out=Path(OUTPUT)
df=pd.read_csv(out/'comparison.csv')
display(df[['model','stage','score','mae','std','p95','max_error']].round(5))
for name in ['pi-edfa','transformer']:
    f=out/name/'results.json'
    r=json.loads(f.read_text())
    print('\n',name,'\n',
          'recent Train Transfer:',r['transfer_val']['score'],
          'recent Train Scratch:',r.get('scratch_val',{}).get('score','not run'))
    if 'transfer_testground' in r:
        display(pd.DataFrame([{'category':k,**v} for k,v in
                              r['transfer_testground'].items()]).round(5))

## 5. 비교 그래프 및 결과 ZIP 저장

학습 기록, 검증 지표, Stage별 가중치(.pt), 제출 형식의 95채널 예측 CSV가 포함됩니다.

In [ ]:
import matplotlib.pyplot as plt
for name in ['pi-edfa','transformer']:
    plt.figure(figsize=(8,4))
    for stage in ['cosmos_pretrain','recent_finetune','recent_scratch']:
        h=out/name/(stage+'_history.csv')
        if h.exists():
            d=pd.read_csv(h)
            plt.plot(d['epoch'],d['val_score'],label=stage)
    plt.title(name+' validation score by stage')
    plt.xlabel('Epoch');plt.ylabel('Official-style validation score')
    plt.grid(alpha=.25);plt.legend();plt.show()
import shutil
archive=shutil.make_archive('/content/ofc_cosmos_transfer_results','zip',str(out))
print('ZIP created:',archive)
from google.colab import files
files.download(archive)